# Function Calling: Агент-художник с Turtle Graphics

В этом ноутбуке мы научимся использовать **Function Calling** — механизм, позволяющий языковой модели вызывать внешние функции для выполнения задач.

## Задача

Мы создадим агента, который умеет рисовать с помощью библиотеки **jturtle** (Jupyter Turtle Graphics). Агент будет:
1. Получать описание того, что нужно нарисовать (например, "дом")
2. Вызывать функции движения черепашки: `forward`, `left`, `right`, `penup`, `pendown`
3. Создавать рисунок по своему "пониманию" задачи

Начнём с установки необходимых библиотек:

In [ ]:
%pip install --upgrade openai python-dotenv jturtle

**ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

In [ ]:
!curl -o .env {{url_of_dotenv_file}}\n

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

def printx(string):
    display(Markdown(string))

print(f"✅ Авторизация настроена (folder_id: {folder_id[:8]}...)")

---

## Часть 1: Знакомство с jturtle

Сначала посмотрим, как работает библиотека jturtle. Это простая черепашья графика, где мы управляем "черепашкой", которая оставляет след при движении.

In [ ]:
import jturtle as turtle

# Рисуем треугольник
turtle.forward(100)   # Движение вперёд на 100 пикселей
turtle.right(120)     # Поворот направо на 120 градусов
turtle.forward(100)
turtle.right(120)
turtle.forward(100)
turtle.right(120)
turtle.done()         # Показать результат

Доступные команды:
- `turtle.forward(distance)` — движение вперёд
- `turtle.left(degrees)` — поворот налево
- `turtle.right(degrees)` — поворот направо
- `turtle.penup()` — поднять перо (движение без рисования)
- `turtle.pendown()` — опустить перо (продолжить рисовать)
- `turtle.done()` — показать результат

---

## Часть 2: Настройка OpenAI клиента

Создадим клиент для работы с Yandex Cloud через OpenAI-совместимый API:

In [ ]:
# Создайте клиент для работы с LLM
# Убедитесь, что он работает

---

## Часть 3: Определение функций для Function Calling

Чтобы модель могла вызывать функции черепашки, нам нужно описать их в специальном формате. Function Calling работает так:

1. Мы описываем доступные функции в виде JSON-схемы
2. Модель анализирует запрос и решает, какие функции вызвать
3. Мы получаем от модели инструкции по вызову функций
4. Выполняем функции и возвращаем результат модели

### Описание инструментов (tools)

Каждый tool — это JSON-объект с описанием функции и её параметров:

In [ ]:
# Описание доступных функций черепашки
turtle_tools = [
    {
        "type": "function",
        "name": "forward",
        "description": "Переместить черепашку вперёд на указанное расстояние в пикселях. При опущенном пере оставляет линию.",
        "parameters": {
            "type": "object",
            "properties": {
                "distance": {
                    "type": "number",
                    "description": "Расстояние в пикселях (обычно от 10 до 200)"
                }
            },
            "required": ["distance"]
        }
    },
    {
        "type": "function",
        "name": "left",
        "description": "Повернуть черепашку налево на указанный угол в градусах.",
        "parameters": {
            "type": "object",
            "properties": {
                "degrees": {
                    "type": "number",
                    "description": "Угол поворота в градусах (например, 90 для прямого угла)"
                }
            },
            "required": ["degrees"]
        }
    },
    {
        "type": "function",
        "name": "right",
        "description": "Повернуть черепашку направо на указанный угол в градусах.",
        "parameters": {
            "type": "object",
            "properties": {
                "degrees": {
                    "type": "number",
                    "description": "Угол поворота в градусах (например, 90 для прямого угла)"
                }
            },
            "required": ["degrees"]
        }
    },
    {
        "type": "function",
        "name": "penup",
        "description": "Поднять перо. После этого черепашка будет двигаться без рисования линий.",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "type": "function",
        "name": "pendown",
        "description": "Опустить перо. После этого черепашка будет рисовать линии при движении.",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    }
]

print(f"✅ Определено {len(turtle_tools)} функций для черепашки")

---

## Часть 4: Простой вызов с Function Calling

Теперь попробуем отправить запрос модели с описанием функций и посмотрим, как она ответит:

In [ ]:
# Напиши системный промпт для модели. Задача - вызывая инструменты
# нарисовать то, о чем просит пользователь
# Посмотрите, что вернёт модель на простой запрос нарисовать квадрат

Модель должна вернуть **function_call** — это означает, что она хочет вызвать функцию. Теперь нам нужно:
1. Выполнить эту функцию
2. Вернуть результат модели
3. Продолжить диалог, пока модель не завершит рисование

---

## Часть 5: Обработка вызовов функций

Создадим функцию-диспетчер, которая выполняет команды черепашки:

In [ ]:
# Реализуйте функцию-диспетчер, которая будет вызывать
# функции из библиотеки `jturtle` по результату, возвращенному
# LLM

---

## Часть 6: Полный цикл рисования

Теперь соберём всё вместе: отправляем запрос, обрабатываем вызовы функций в цикле, пока модель не закончит рисование.

In [ ]:
# Реализуйте функцию `draw_with_agent`, которая:
# - вызывает LLM со списком инструментов
# - обрабатывает ответ и вызывает функцию-диспетчер для каждого function tool

def draw_with_agent(req):
    pass

### Тест: рисуем квадрат

In [ ]:
draw_with_agent("Нарисуй квадрат со стороной 100 пикселей")

### Тест: рисуем дом

In [ ]:
draw_with_agent("Нарисуй простой дом: квадрат с треугольной крышей")

---

## Часть 7: Использование Pydantic для описания функций

В предыдущем примере мы описывали функции вручную в виде JSON. Это работает, но для больших проектов удобнее использовать **Pydantic** — библиотеку для валидации данных, которая автоматически генерирует JSON-схему.

Преимущества Pydantic:
- Автоматическая генерация схемы из типов Python
- Валидация входных данных
- Удобное описание через docstring и Field

In [ ]:
# Реализуйте классы для каждой из команд черепашки:
# - Наследуйте классы от pydantic.BaseModel
# - Внутри класса опишите поля-параметры команды (если есть)
# - Опишите для каждого класса функцию `execute`, которая выполняет команду

---

## Часть 8: Класс Agent

Теперь создадим универсальный класс `Agent`, который:
- Автоматически генерирует описания инструментов из Pydantic-моделей
- Обрабатывает вызовы функций в цикле
- Управляет историей диалога

Для вашего удобства мы приводим реализацию этого класса. Изучите код, убедитесь, что вы его понимаете:

In [ ]:
class Agent:
    """
    Универсальный агент с поддержкой Function Calling.
    """
    
    def __init__(self, instruction: str, tools: list, model: str = model, max_iterations: int = 50):
        self.instruction = instruction
        self.model = model
        self.max_iterations = max_iterations
        
        # Создаём словарь имя -> класс для быстрого поиска
        self.tool_map = {cls.__name__: cls for cls in tools}
        
        # Генерируем описания инструментов для API
        self.tools_schema = [
            {
                "type": "function",
                "name": cls.__name__,
                "description": cls.__doc__ or "",
                "parameters": cls.model_json_schema()
            }
            for cls in tools
        ]
        
        # История сессий
        self.sessions = {}
    
    def reset_sessions(self):
        self.sessions = {}

    def __call__(self, message: str, session_id: str = "default", verbose: bool = True) -> str:
        """
        Обработка сообщения пользователя с автоматическим выполнением функций.
        """
        # Получаем или создаём сессию
        session = self.sessions.get(session_id, {"last_response_id": None})
        
        # Первый запрос
        res = client.responses.create(
            model=self.model,
            tools=self.tools_schema,
            instructions=self.instruction,
            reasoning = { "effort" : "high" },
            previous_response_id=session["last_response_id"],
            store=True,
            input=message
        )
        
        # Цикл обработки функций
        for _ in range(self.max_iterations):
            tool_calls = [item for item in res.output if item.type == "function_call"]
            
            if not tool_calls:
                break
            
            # Обрабатываем вызовы
            outputs = []
            for call in tool_calls:
                if verbose:
                    print(f"  🔧 {call.name}({call.arguments})")
                
                try:
                    # Получаем класс и создаём объект
                    cls = self.tool_map[call.name]
                    args = json.loads(call.arguments) if call.arguments else {}
                    obj = cls.model_validate(args)
                    
                    # Выполняем функцию
                    result = obj.execute()
                except Exception as e:
                    result = f"Ошибка: {e}"
                
                outputs.append({
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": result
                })
            
            # Отправляем результаты
            res = client.responses.create(
                model=self.model,
                tools=self.tools_schema,
                previous_response_id=res.id,
                store=True,
                input=outputs
            )
        
        # Сохраняем состояние сессии
        session["last_response_id"] = res.id
        self.sessions[session_id] = session
        
        return res.output_text or "Готово!"

print("✅ Класс Agent определён")

---

## Часть 9: Создание агента-художника

Теперь создадим агента с инструкцией для рисования:

In [ ]:
# Создайте агента-художника `artist`, задав ему подробный системный
# промпт и передав список инструментов

---

## Часть 10: Рисуем с помощью агента

Функция-обёртка для удобного рисования:

In [ ]:
import jturtle as turtle

def draw(prompt: str):
    """Попросить агента нарисовать что-то."""
    print(f"🎨 Задание: {prompt}")
    print("="*50)
    
    result = artist(prompt)
    
    print("="*50)
    print(f"💬 {result}")
    turtle.done()

### Примеры рисования

In [ ]:
# Рисуем дом
draw("Нарисуй дом: квадрат с треугольной крышей сверху")

In [ ]:
# Рисуем дерево
draw("Нарисуй дерево")

In [ ]:
# Рисуем человечка
draw("Нарисуй лицо человека")

In [ ]:
# Рисуем звезду
draw("Нарисуй пятиконечную звезду")

---

## Заключение

В этом ноутбуке мы изучили **Function Calling** — мощный механизм, позволяющий LLM выполнять действия во внешнем мире.

### Что мы узнали:

1. **Описание инструментов** — как создать JSON-схему для функций
2. **Цикл обработки** — как получать вызовы от модели и возвращать результаты
3. **Pydantic-модели** — удобный способ описания функций с валидацией
4. **Класс Agent** — универсальная обёртка для работы с Function Calling

### Применение Function Calling:

- 🔧 **Инструменты** — калькуляторы, поиск, API-вызовы
- 🤖 **Автоматизация** — управление системами, DevOps
- 🎮 **Игры** — управление персонажами, генерация контента
- 📊 **Аналитика** — запросы к базам данных, построение графиков

### Дальнейшее изучение:

- Добавьте новые команды черепашки (цвет, толщина линии)
- Создайте агента для работы с файлами или API
- Попробуйте комбинировать несколько агентов